In [1]:
import itertools
import os
import time
from joblib import Parallel, delayed
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from numpy import mean, sqrt, zeros, zeros_like
from numpy.linalg import cond, det
import pandas as pd
from sklearn.datasets import make_regression

# Use non-interactive backend for thread-safe parallel plot generation
matplotlib.use('Agg')


1. OPTIMIZERS

In [2]:
class ADAM:
    '''
    Adaptive Moment Estimation (Adam) optimizer with first and second
    moment bias correction.
    '''

    def __init__(self, learning_rate, b1=0.9, b2=0.999):
        '''
        Initializes the Adam optimizer hyperparameters and state buffers.

        Args:
            learning_rate (float): Step size alpha.
            b1 (float): Exponential decay rate for the first moment estimates.
            b2 (float): Exponential decay rate for the second moment estimates.
        '''
        self.learning_rate = learning_rate
        self.b1 = b1
        self.b2 = b2
        self.mw = None
        self.vw = None
        self.t = 0
        self.epsilon = 1e-8

    def step(self, w, dw):
        '''
        Performs a single optimization step using bias-corrected moments.

        Args:
            w (np.ndarray): Current weight vector of shape (d, 1).
            dw (np.ndarray): Current gradient vector of shape (d, 1).

        Returns:
            np.ndarray: Updated weight vector of shape (d, 1).
        '''
        # Lazily allocate momentum buffers matching the weight dimensions
        if self.mw is None:
            self.mw = zeros_like(w)
            self.vw = zeros_like(w)

        self.t += 1

        # Update biased first moment (mean) and second raw moment (uncentered variance)
        self.mw = self.b1 * self.mw + (1.0 - self.b1) * dw
        self.vw = self.b2 * self.vw + (1.0 - self.b2) * (dw ** 2)

        # Counteract zero-initialization bias during early iterations
        mw_h = self.mw / (1.0 - self.b1 ** self.t)
        vw_h = self.vw / (1.0 - self.b2 ** self.t)

        # Apply adaptive coordinate-wise parameter update
        w -= (self.learning_rate / (sqrt(vw_h) + self.epsilon)) * mw_h
        return w


In [3]:
class OLS:
    '''
    Analytical Ordinary Least Squares (OLS) solver using the
    Moore-Penrose pseudoinverse.
    '''

    def step(self, x_mat, y_vec):
        '''
        Computes the minimum-norm least-squares solution w = pinv(X) @ y.

        Args:
            x_mat (np.ndarray): Design matrix of shape (N, d).
            y_vec (np.ndarray): Target vector of shape (N, 1).

        Returns:
            np.ndarray: Optimal weight vector of shape (d, 1).
        '''
        # SVD-based pseudoinverse handles rank-deficient or ill-conditioned matrices
        x_ext = np.linalg.pinv(x_mat)
        return x_ext @ y_vec

In [4]:
class NAG:
    '''
    Nesterov Accelerated Gradient (NAG) optimizer.
    Evaluates gradients at the predicted lookahead position.
    '''

    def __init__(self, learning_rate, gamma=0.9):
        '''
        Initializes NAG hyperparameters and velocity state.

        Args:
            learning_rate (float): Step size for gradient descent.
            gamma (float): Momentum retention coefficient in [0, 1).
        '''
        self.learning_rate = learning_rate
        self.gamma = gamma
        self.v_w = None

    def get_lookahead(self, w):
        '''
        Projects weights forward along the accumulated velocity vector
        prior to gradient evaluation.

        Args:
            w (np.ndarray): Current weight vector of shape (d, 1).

        Returns:
            np.ndarray: Lookahead weight vector of shape (d, 1).
        '''
        if self.v_w is None:
            self.v_w = np.zeros_like(w)
        return w - self.gamma * self.v_w

    def step(self, w, dw):
        '''
        Updates the velocity buffer using the lookahead gradient and moves weights.

        Args:
            w (np.ndarray): Current weight vector of shape (d, 1).
            dw (np.ndarray): Gradient evaluated at the lookahead position.

        Returns:
            np.ndarray: Updated weight vector of shape (d, 1).
        '''
        if self.v_w is None:
            self.v_w = np.zeros_like(w)

        # Accumulate velocity and apply update
        self.v_w = self.gamma * self.v_w + self.learning_rate * dw
        w -= self.v_w
        return w

In [5]:
class DREM:
    '''
    Dynamic Regressor Extension and Mixing (DREM) optimizer.
    Transforms a d-dimensional coupled regression problem into d independent
    scalar regressions via adjugate matrix multiplication and Cramer's rule.
    '''

    def __init__(self, alpha=0.01, mode='raw'):
        '''
        Initializes the DREM optimizer.

        Args:
            alpha (float): Adaptation gain (learning rate).
            mode (str): Batch formation strategy ('raw', 'direct',
                        'gersh_exit', 'gersh_cond', 'seq_d_opt',
                        'seq_dpp', 'seq_cos').
        '''
        self.alpha = alpha
        self.mode = mode

    def _compute_cramer(self, x_b, y_b):
        '''
        Computes the scalar regressor excitation delta = det(X_b) and the
        decoupled target vector Y = adj(X_b) @ y_b using Cramer's rule.

        Args:
            x_b (np.ndarray): Square mini-batch feature matrix of shape (d, d).
            y_b (np.ndarray): Mini-batch target vector of shape (d, 1).

        Returns:
            tuple: (delta, y_vector) where delta is float and y_vector is (d, 1).
        '''
        delta = det(x_b)
        size = x_b.shape[0]
        y_vector = zeros((size, 1))

        # By Cramer's rule, the i-th component of adj(X_b) @ y_b equals the
        # determinant of X_b with its i-th column replaced by y_b
        for i in range(size):
            x_i = x_b.copy()
            x_i[:, i] = y_b.flatten()
            y_vector[i] = det(x_i)

        return delta, y_vector

    def step_direct(self, weights, x_b, y_b, rho=0.5, eps_delta=1e-6):
        '''
        Point-wise algebraic estimator w_dir = Y / delta coupled with a
        first-order low-pass filter to attenuate measurement noise.

        Args:
            weights (np.ndarray): Current parameter vector of shape (d, 1).
            x_b (np.ndarray): Square mini-batch regressor of shape (d, d).
            y_b (np.ndarray): Mini-batch target vector of shape (d, 1).
            rho (float): Low-pass filter pole in [0, 1) for exponential smoothing.
            eps_delta (float): Singularity threshold for determinant magnitude.

        Returns:
            np.ndarray: Updated parameter vector of shape (d, 1).
        '''
        delta, y_vector = self._compute_cramer(x_b, y_b)

        # Reject update if the batch is nearly singular or numerically unstable
        if np.abs(delta) < eps_delta or np.abs(delta) > 1e6:
            return weights

        # Direct point-wise inversion of the decoupled scalar regressions
        w_dir = y_vector / delta

        # First-order low-pass filtering: w_k = rho * w_{k-1} + (1 - rho) * w_dir
        return rho * weights + (1.0 - rho) * w_dir

    def step(self, weights, x_b, y_b):
        '''
        Normalized gradient descent update on decoupled scalar regressions.

        Args:
            weights (np.ndarray): Current parameter vector of shape (d, 1).
            x_b (np.ndarray): Square mini-batch regressor of shape (d, d).
            y_b (np.ndarray): Mini-batch target vector of shape (d, 1).

        Returns:
            np.ndarray: Updated parameter vector of shape (d, 1).
        '''
        delta, y_vector = self._compute_cramer(x_b, y_b)

        # Guard against division by zero or float overflow in delta ** 2
        if np.abs(delta) < 1e-6 or np.abs(delta) > 1e6:
            return weights

        alpha_eff = self.alpha

        # Normalized step guarantees monotonic element-wise error decay in noise-free case
        numerator = alpha_eff * delta * (delta * weights - y_vector)
        denominator = 1.0 + alpha_eff * (delta ** 2)
        weights -= numerator / denominator
        return weights

2. BATCH SELECTION LOGIC

In [6]:
def get_batch_candidates(xs, ys, start_idx, bs, pool_factor=4):
    '''
    Extracts a local candidate window of size (bs * pool_factor) from the dataset
    to limit the computational overhead of greedy subset selection.

    Args:
        xs (np.ndarray): Shuffled feature matrix of shape (N, d).
        ys (np.ndarray): Shuffled target vector of shape (N, 1).
        start_idx (int): Starting row index for the candidate window.
        bs (int): Target batch size (equal to feature dimension d).
        pool_factor (int): Multiplier determining candidate pool size.

    Returns:
        tuple: (x_pool, y_pool) candidate sub-arrays.
    '''
    n_samples = xs.shape[0]
    end_idx = min(start_idx + bs * pool_factor, n_samples)
    return xs[start_idx:end_idx], ys[start_idx:end_idx]


def select_seq_gersh_batch(x_pool, y_pool, bs, criterion='min_eig'):
    '''
    Greedily constructs a batch of size bs by optimizing Gershgorin disc
    locations of the information (Gram) matrix A = X_trial^T @ X_trial.

    By Gershgorin's circle theorem, every eigenvalue of A lies within at least
    one interval [c_i - r_i, c_i + r_i], where c_i = A_{ii} and
    r_i = sum_{j != i} |A_{ij}|.

    Args:
        x_pool (np.ndarray): Candidate feature pool of shape (M, d).
        y_pool (np.ndarray): Candidate target pool of shape (M, 1).
        bs (int): Required batch size d.
        criterion (str): 'min_eig' to maximize the lower spectral bound
                         min_i(c_i - r_i); 'cond' to minimize the Gershgorin
                         condition number proxy max_i(c_i + r_i) / min_i(c_i - r_i).

    Returns:
        tuple: Selected (x_batch, y_batch) of shapes (d, d) and (d, 1),
               or (None, None) if pool is insufficient.
    '''
    n_pool = x_pool.shape[0]
    if n_pool < bs:
        return None, None

    selected = []
    remaining = list(range(n_pool))

    # Seed the batch with the sample having the highest signal energy (L2 norm squared)
    norms = [np.dot(x_pool[r], x_pool[r]) for r in remaining]
    first_idx = remaining[np.argmax(norms)]
    selected.append(first_idx)
    remaining.remove(first_idx)

    # Greedily add (bs - 1) samples that optimize the chosen Gershgorin bound
    for _ in range(bs - 1):
        best_cand = None
        best_metric = -np.inf if criterion == 'min_eig' else np.inf

        for cand in remaining:
            trial_indices = selected + [cand]
            x_trial = x_pool[trial_indices]

            # Form Gram matrix A = X_trial^T @ X_trial of shape (d, d)
            a_gram = x_trial.T @ x_trial
            c_centers = np.diag(a_gram)
            r_radii = np.sum(np.abs(a_gram), axis=1) - c_centers

            # Worst-case lower and upper eigenvalue bounds across all d discs
            min_bound = np.min(c_centers - r_radii)
            max_bound = np.max(c_centers + r_radii)

            if criterion == 'min_eig':
                # Push the leftmost disc boundary away from zero to prevent singularity
                metric = min_bound
                if metric > best_metric:
                    best_metric = metric
                    best_cand = cand
            else:
                # Minimize the ratio of upper to lower Gershgorin bounds
                if min_bound > 1e-6:
                    metric = max_bound / min_bound
                else:
                    # Heavy penalty when any Gershgorin disc encompasses zero
                    metric = 1e6 + (max_bound - min_bound)

                if metric < best_metric:
                    best_metric = metric
                    best_cand = cand

        # Fallback to first available sample if all candidates yield identical metrics
        if best_cand is None:
            best_cand = remaining[0]

        selected.append(best_cand)
        remaining.remove(best_cand)

    return x_pool[selected], y_pool[selected]

In [7]:
def select_seq_d_opt_batch(x_pool, y_pool, bs, reg=1e-4):
    '''
    Sequentially constructs a D-optimal batch that maximizes the log-determinant
    of the regularized information matrix det(X_b^T @ X_b + reg * I).
    Uses rank-1 Sherman-Morrison updates to maintain the inverse in O(d^2) time.

    Args:
        x_pool (np.ndarray): Candidate feature pool of shape (M, d).
        y_pool (np.ndarray): Candidate target pool of shape (M, 1).
        bs (int): Required batch size d.
        reg (float): Tikhonov regularization parameter for initial matrix inversion.

    Returns:
        tuple: Selected (x_batch, y_batch) or (None, None).
    '''
    n_pool = x_pool.shape[0]
    if n_pool < bs:
        return None, None

    d = x_pool.shape[1]
    # Initialize inverse covariance matrix M_0^{-1} = (reg * I)^{-1}
    m_inv = np.eye(d) / reg
    selected = []
    remaining = list(range(n_pool))

    for _ in range(bs):
        best_cand = -1
        best_h = -1.0

        # By Sylvester's determinant theorem, det(M + x x^T) = det(M) * (1 + x^T M^{-1} x)
        # Maximizing the determinant increment is equivalent to maximizing leverage h_i
        for cand in remaining:
            x_i = x_pool[cand:cand + 1].T
            h_i = (x_i.T @ m_inv @ x_i)[0, 0]
            if h_i > best_h:
                best_h = h_i
                best_cand = cand

        if best_cand == -1:
            return None, None

        selected.append(best_cand)
        remaining.remove(best_cand)

        # Update M^{-1} via Sherman-Morrison formula: M_{k+1}^{-1} = M_k^{-1} - (M_k^{-1} x x^T M_k^{-1}) / (1 + h)
        x_best = x_pool[best_cand:best_cand + 1].T
        m_inv = m_inv - (m_inv @ x_best @ x_best.T @ m_inv) / (1.0 + best_h)

    return x_pool[selected], y_pool[selected]

In [8]:
def select_seq_dpp_batch(x_pool, y_pool, bs):
    '''
    Samples a diverse mini-batch using a sequential k-Determinantal Point
    Process (DPP) via iterative Gram-Schmidt orthogonalization.

    Args:
        x_pool (np.ndarray): Candidate feature pool of shape (M, d).
        y_pool (np.ndarray): Candidate target pool of shape (M, 1).
        bs (int): Required batch size d.

    Returns:
        tuple: Selected (x_batch, y_batch) or (None, None).
    '''
    n_pool = x_pool.shape[0]
    if n_pool < bs:
        return None, None

    selected = []
    remaining = list(range(n_pool))
    basis = []

    for _ in range(bs):
        probs = np.zeros(len(remaining))

        # Compute squared distance of each candidate to the subspace spanned by 'basis'
        for idx, cand in enumerate(remaining):
            x_i = x_pool[cand]
            x_perp = x_i.copy()

            # Subtract projections onto all previously orthogonalized basis vectors
            for v in basis:
                v_norm_sq = np.dot(v, v)
                if v_norm_sq > 1e-12:
                    x_perp -= (np.dot(x_i, v) / v_norm_sq) * v

            # Volume contribution is proportional to the squared orthogonal residual
            probs[idx] = np.dot(x_perp, x_perp)

        total_p = np.sum(probs)
        # Abort if remaining candidates are linearly dependent on the current basis
        if total_p < 1e-12:
            return None, None

        # Sample next vector proportionally to its squared orthogonal component
        chosen_idx = np.random.choice(len(remaining), p=probs / total_p)
        chosen_cand = remaining[chosen_idx]

        # Compute orthogonal residual of the chosen vector to expand the basis
        x_chosen = x_pool[chosen_cand]
        x_perp_chosen = x_chosen.copy()
        for v in basis:
            v_norm_sq = np.dot(v, v)
            if v_norm_sq > 1e-12:
                x_perp_chosen -= (np.dot(x_chosen, v) / v_norm_sq) * v

        basis.append(x_perp_chosen)
        selected.append(chosen_cand)
        remaining.pop(chosen_idx)

    return x_pool[selected], y_pool[selected]

In [9]:
def select_seq_cos_batch(x_pool, y_pool, bs):
    '''
    Greedily constructs a batch of mutually near-orthogonal vectors by
    minimizing the maximum absolute cosine similarity with already selected samples.

    Args:
        x_pool (np.ndarray): Candidate feature pool of shape (M, d).
        y_pool (np.ndarray): Candidate target pool of shape (M, 1).
        bs (int): Required batch size d.

    Returns:
        tuple: Selected (x_batch, y_batch) or (None, None).
    '''
    n_pool = x_pool.shape[0]
    if n_pool < bs:
        return None, None

    selected = []
    remaining = list(range(n_pool))

    # Start with the vector of largest Euclidean norm
    norms = [np.linalg.norm(x_pool[r]) for r in remaining]
    best_first = remaining[np.argmax(norms)]
    selected.append(best_first)
    remaining.remove(best_first)

    for _ in range(bs - 1):
        best_cand = None
        min_max_cos = float('inf')

        for cand in remaining:
            x_cand = x_pool[cand]
            norm_cand = np.linalg.norm(x_cand)
            if norm_cand < 1e-8:
                continue

            # Find the highest collinearity between candidate and current batch members
            max_cos_for_cand = -1.0
            for sel in selected:
                x_sel = x_pool[sel]
                norm_sel = np.linalg.norm(x_sel)
                if norm_sel < 1e-8:
                    continue

                cos_sim = abs(np.dot(x_cand, x_sel) / (norm_cand * norm_sel))
                if cos_sim > max_cos_for_cand:
                    max_cos_for_cand = cos_sim

            # Select candidate that minimizes worst-case collinearity (Min-Max angle criterion)
            if max_cos_for_cand < min_max_cos:
                min_max_cos = max_cos_for_cand
                best_cand = cand

        if best_cand is not None:
            selected.append(best_cand)
            remaining.remove(best_cand)
        else:
            return None, None

    return x_pool[selected], y_pool[selected]


3. BATCH DISTRIBUTION ANALYSIS

In [10]:
def evaluate_batch_distributions(x_raw, y_raw, folder_name, dataset_name,
                                 n_epochs=10, seed=42):
    '''
    Evaluates and plots the empirical distribution of batch informativeness
    I(X_b) = log10(|det(X_b)| + 1e-12) across all batch selection strategies.
    Generates a horizontal histogram with Informativeness on the Y-axis and
    Batch Count on the X-axis, exporting both PNG plots and CSV tables.

    Args:
        x_raw (np.ndarray): Raw feature matrix of shape (N, d - 1).
        y_raw (np.ndarray): Raw target vector of shape (N, 1).
        folder_name (str): Output directory path.
        dataset_name (str): Identifier used for output filenames and titles.
        n_epochs (int): Number of random passes over the dataset to collect statistics.
        seed (int): Random seed for reproducible shuffling.
    '''
    os.makedirs(folder_name, exist_ok=True)
    rng = np.random.RandomState(seed)

    # Standardize features and targets to zero mean and unit variance
    x_mean, x_std = x_raw.mean(axis=0), x_raw.std(axis=0)
    x_norm = (x_raw - x_mean) / (x_std + 1e-8)
    y_mean, y_std = y_raw.mean(axis=0), y_raw.std(axis=0)
    y_norm = (y_raw - y_mean) / (y_std + 1e-8)

    # Prepend column of ones to account for the intercept (bias) parameter
    x_ext = np.hstack([np.ones((x_norm.shape[0], 1)), x_norm])

    n_samples, bs = x_ext.shape
    methods = ['raw', 'gersh_exit', 'gersh_cond', 'seq_d_opt', 'seq_dpp', 'seq_cos']
    info_records = {m: [] for m in methods}

    # Collect batch log-determinants across multiple epochs
    for _ in range(n_epochs):
        idx = rng.permutation(n_samples)
        xs, ys = x_ext[idx], y_norm[idx]

        for m in methods:
            i = 0
            while i <= n_samples - bs:
                if m == 'seq_d_opt':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, _ = select_seq_d_opt_batch(x_pool, y_pool, bs)
                elif m == 'seq_dpp':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, _ = select_seq_dpp_batch(x_pool, y_pool, bs)
                elif m == 'seq_cos':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, _ = select_seq_cos_batch(x_pool, y_pool, bs)
                elif m == 'gersh_exit':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, _ = select_seq_gersh_batch(
                        x_pool, y_pool, bs, criterion='min_eig'
                    )
                elif m == 'gersh_cond':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, _ = select_seq_gersh_batch(
                        x_pool, y_pool, bs, criterion='cond'
                    )
                else:
                    xb = xs[i:i + bs]

                if xb is None:
                    xb = xs[i:i + bs]

                i += bs

                # Compute log10(|det(X_b)|) as a measure of D-optimal batch volume
                abs_det = np.abs(det(xb))
                log_info = np.log10(abs_det + 1e-12)
                info_records[m].append(log_info)

    # Build a shared bin grid spanning the global min/max log-determinant values
    all_vals = np.concatenate(
        [info_records[m] for m in methods if len(info_records[m]) > 0]
    )
    bins = np.linspace(np.min(all_vals), np.max(all_vals), 35)
    bin_centers = 0.5 * (bins[:-1] + bins[1:])

    # Export frequency counts and empirical probabilities to CSV
    csv_dict = {'bin_center_log10_det': bin_centers}
    for m in methods:
        counts, _ = np.histogram(info_records[m], bins=bins)
        probs = counts / max(1, np.sum(counts))
        csv_dict[f'count_{m}'] = counts
        csv_dict[f'prob_{m}'] = probs

    df_dist = pd.DataFrame(csv_dict)
    csv_path = os.path.join(folder_name, f'batch_distribution_{dataset_name}.csv')
    df_dist.to_csv(csv_path, index=False)

    # Plot horizontal histograms: Y-axis = Informativeness, X-axis = Batch Count
    fig, axes = plt.subplots(1, len(methods), figsize=(18, 6), sharey=True)
    colors = {
        'raw': 'red',
        'gersh_exit': 'purple',
        'gersh_cond': 'brown',
        'seq_d_opt': 'olive',
        'seq_dpp': 'teal',
        'seq_cos': 'magenta'
    }

    for ax, m in zip(axes, methods):
        vals = info_records[m]
        ax.hist(
            vals,
            bins=bins,
            orientation='horizontal',
            color=colors[m],
            alpha=0.7,
            edgecolor='black'
        )
        mean_val = np.mean(vals)
        ax.axhline(
            y=mean_val,
            color='black',
            linestyle='--',
            linewidth=1.5,
            label=f'Mean: {mean_val:.2f}'
        )
        ax.set_title(f'{m}\n(N={len(vals)})')
        ax.set_xlabel('Count (Batches)')
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='lower right', fontsize=8)

    axes[0].set_ylabel('Informativeness $\\log_{10}|\\det(X_b)|$')
    fig.suptitle(
        f'Batch Informativeness Distribution | Dataset: {dataset_name}',
        fontsize=14
    )
    plt.tight_layout()

    plot_path = os.path.join(folder_name, f'batch_distribution_{dataset_name}.png')
    fig.savefig(plot_path)
    plt.close(fig)
    print(f'Saved batch distribution to: {plot_path} and {csv_path}')

4. TRAINING LOOP (TWO-STAGE)

In [11]:
def train(x_ext, y, optimizer_obj, w_init, polish_type='adam',
          polish_threshold=0.5, epochs=30, polish_lr=0.01, seed=None):
    '''
    Executes a two-stage hybrid training procedure:
    - Stage 1 (Rapid Descent): Uses DREM (either gradient or direct point-wise
      estimation) with active batch selection until MSE drops below polish_threshold.
    - Stage 2 (Asymptotic Polishing): Switches to Adam or NAG to filter out
      residual stochastic noise without computing O(d^4) determinants.

    Args:
        x_ext (np.ndarray): Standardized design matrix with bias of shape (N, d).
        y (np.ndarray): Standardized target vector of shape (N, 1).
        optimizer_obj (object): Primary optimizer instance (DREM, ADAM, NAG, or OLS).
        w_init (np.ndarray): Initial weight vector of shape (d, 1) at epoch 0.
        polish_type (str): Secondary optimizer type ('adam' or 'nag').
        polish_threshold (float): MSE threshold triggering switch to Stage 2.
        epochs (int): Total number of training epochs.
        polish_lr (float): Learning rate for the polishing optimizer.
        seed (int, optional): Random seed for epoch-level data shuffling.

    Returns:
        tuple: (mse_history, mae_history, cond_history, switched_epoch).
    '''
    n_samples, n_features_ext = x_ext.shape
    if seed is not None:
        np.random.seed(seed)

    # Copy shared initial weights so all optimizers start from the exact same point
    w = w_init.copy()
    pred_init = x_ext @ w
    mse_0 = mean((y - pred_init) ** 2)
    mae_0 = mean(np.abs(y - pred_init))

    # Record epoch 0 metrics prior to any parameter updates
    mse_history = [mse_0]
    mae_history = [mae_0]
    cond_history = []

    # Analytical baseline returns constant error across all epochs
    if isinstance(optimizer_obj, OLS):
        w = optimizer_obj.step(x_ext, y)
        pred = x_ext @ w
        mse_val = mean((y - pred) ** 2)
        mae_val = mean(np.abs(y - pred))
        return [mse_0] + [mse_val] * epochs, [mae_0] + [mae_val] * epochs, [], 0

    # Instantiate secondary gradient-based polisher
    if polish_type == 'adam':
        polisher = ADAM(learning_rate=polish_lr)
    else:
        polisher = NAG(learning_rate=polish_lr)

    # Determine initial operating stage
    if isinstance(optimizer_obj, DREM):
        stage = 'direct' if optimizer_obj.mode == 'direct' else 'drem'
    else:
        stage = 'polish'

    switched_epoch = None

    for epoch in range(epochs):
        idx = np.random.permutation(n_samples)
        xs, ys = x_ext[idx], y[idx]
        bs = n_features_ext
        cond_epoch = []

        i = 0
        while i <= n_samples - bs:
            if stage in ('direct', 'drem'):
                mode = optimizer_obj.mode
                if mode == 'seq_d_opt':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, yb = select_seq_d_opt_batch(x_pool, y_pool, bs)
                elif mode == 'seq_dpp':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, yb = select_seq_dpp_batch(x_pool, y_pool, bs)
                elif mode == 'seq_cos':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, yb = select_seq_cos_batch(x_pool, y_pool, bs)
                elif mode == 'gersh_exit':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, yb = select_seq_gersh_batch(
                        x_pool, y_pool, bs, criterion='min_eig'
                    )
                elif mode == 'gersh_cond':
                    x_pool, y_pool = get_batch_candidates(xs, ys, i, bs)
                    xb, yb = select_seq_gersh_batch(
                        x_pool, y_pool, bs, criterion='cond'
                    )
                else:
                    # Modes 'raw' and 'direct' use standard contiguous mini-batches
                    xb, yb = xs[i:i + bs], ys[i:i + bs]

                # Fallback to contiguous slice if candidate pool is smaller than bs
                if xb is None:
                    xb, yb = xs[i:i + bs], ys[i:i + bs]

                i += bs

                # Track 2-norm condition number of the selected square regressor matrix
                c = cond(xb)
                if np.isfinite(c):
                    cond_epoch.append(c)

                # Execute primary DREM update
                if stage == 'direct':
                    w = optimizer_obj.step_direct(w, xb, yb, rho=0.5)
                else:
                    w = optimizer_obj.step(w, xb, yb)

                # Evaluate full-dataset MSE to check if polishing threshold is reached
                current_mse_step = mean((y - x_ext @ w) ** 2)
                if current_mse_step < polish_threshold:
                    stage = 'polish'
                    if switched_epoch is None:
                        switched_epoch = epoch

            else:
                # Stage 2: Polish weights with Adam or NAG on standard mini-batches
                xb, yb = xs[i:i + bs], ys[i:i + bs]
                i += bs

                active_opt = polisher if isinstance(optimizer_obj, DREM) else optimizer_obj
                if isinstance(active_opt, NAG):
                    w_look = active_opt.get_lookahead(w)
                    pred_b = xb @ w_look
                    dw = (2.0 / bs) * (xb.T @ (pred_b - yb))
                    w = active_opt.step(w, dw)
                else:
                    pred_b = xb @ w
                    dw = (2.0 / bs) * (xb.T @ (pred_b - yb))
                    w = active_opt.step(w, dw)

        # Compute end-of-epoch metrics
        pred = x_ext @ w
        current_mse = mean((y - pred) ** 2)
        current_mae = mean(np.abs(y - pred))

        mse_history.append(current_mse)
        mae_history.append(current_mae)
        cond_history.append(mean(cond_epoch) if cond_epoch else np.nan)

    return mse_history, mae_history, cond_history, switched_epoch

5. EXPERIMENT WORKER

In [12]:
def run_experiment(args):
    '''
    Worker function for parallel execution of a single hyperparameter configuration.
    Standardizes data, runs multiple random seeds from identical initial weights,
    generates convergence plots, and aggregates performance metrics.

    Args:
        args (tuple): (i, combo, folder_name, x_raw, y_raw, title_prefix).

    Returns:
        dict: Aggregated metrics across seeds, or None if execution fails.
    '''
    try:
        i, combo, folder_name, x_raw, y_raw, title_prefix = args
        os.makedirs(folder_name, exist_ok=True)
        if x_raw is None or len(x_raw) == 0:
            return None

        # Z-score standardization of features and targets
        x_mean, x_std = x_raw.mean(axis=0), x_raw.std(axis=0)
        x_norm = (x_raw - x_mean) / (x_std + 1e-8)

        y_mean, y_std = y_raw.mean(axis=0), y_raw.std(axis=0)
        y_norm = (y_raw - y_mean) / (y_std + 1e-8)

        # Append bias column of ones
        x_ext = np.hstack([np.ones((x_norm.shape[0], 1)), x_norm])
        _, d = x_ext.shape

        opts_keys = [
            'drem_direct',
            'drem_raw',
            'drem_gersh_exit',
            'drem_gersh_cond',
            'drem_seq_d_opt',
            'drem_seq_dpp',
            'drem_seq_cos',
            'adam',
            'nag'
        ]

        # Estimate floating-point operations per batch iteration
        # DREM requires (d + 1) LU-based determinants of size d x d -> O(d^4)
        flops_per_iter = {
            'adam': int(4 * d ** 2 + 14 * d),
            'nag': int(4 * d ** 2 + 6 * d),
        }
        for k in opts_keys:
            if 'drem' in k:
                flops_per_iter[k] = int((2 / 3) * d ** 4 + (2 / 3) * d ** 3 + 4 * d)

        polish_optimizer = combo.get('polish_optimizer', 'adam')
        thr_polish = 0.5
        iteration_results = []
        histories = {opt: [] for opt in opts_keys}
        switch_epochs_record = {opt: [] for opt in opts_keys if 'drem' in opt}

        # Run 3 independent trials with shared random initialization per trial
        for j in range(3):
            rng = np.random.RandomState(j)
            w_init = rng.uniform(0.0, 1.0, (d, 1))

            opt_instances = {
                'drem_direct': DREM(alpha=combo['alpha'], mode='direct'),
                'drem_raw': DREM(alpha=combo['alpha'], mode='raw'),
                'drem_gersh_exit': DREM(alpha=combo['alpha'], mode='gersh_exit'),
                'drem_gersh_cond': DREM(alpha=combo['alpha'], mode='gersh_cond'),
                'drem_seq_d_opt': DREM(alpha=combo['alpha'], mode='seq_d_opt'),
                'drem_seq_dpp': DREM(alpha=combo['alpha'], mode='seq_dpp'),
                'drem_seq_cos': DREM(alpha=combo['alpha'], mode='seq_cos'),
                'adam': ADAM(learning_rate=combo['alpha']),
                'nag': NAG(learning_rate=combo['alpha'])
            }

            current_res = {}
            for opt_name, opt_obj in opt_instances.items():
                start = time.perf_counter()

                mse_h, mae_h, cond_h, sw_ep = train(
                    x_ext,
                    y_norm,
                    opt_obj,
                    w_init=w_init,
                    polish_type=polish_optimizer,
                    polish_threshold=thr_polish,
                    epochs=combo['epochs'],
                    polish_lr=combo['alpha'],
                    seed=j
                )

                t_elapsed = time.perf_counter() - start

                if 'drem' in opt_name:
                    switch_epochs_record[opt_name].append(
                        sw_ep if sw_ep is not None else np.nan
                    )

                current_res[f'mse_{opt_name}'] = mse_h[-1]
                current_res[f'mae_{opt_name}'] = mae_h[-1]
                current_res[f'time_{opt_name}'] = t_elapsed
                if cond_h:
                    last_cond = cond_h[-1]
                    current_res[f'cond_{opt_name}'] = (
                        last_cond if np.isfinite(last_cond) else np.nan
                    )

                histories[opt_name].append(mse_h)

            iteration_results.append(current_res)

        # Plot averaged MSE convergence curves
        fig, ax = plt.subplots(figsize=(14, 8))
        styles = {
            'drem_direct': {
                'color': 'cyan', 'linestyle': '-', 'marker': '*', 'markersize': 6
            },
            'drem_raw': {'color': 'red', 'linestyle': '-', 'marker': 'x'},
            'drem_gersh_exit': {'color': 'purple', 'linestyle': '--', 'marker': 'o'},
            'drem_gersh_cond': {'color': 'brown', 'linestyle': '-.', 'marker': 's'},
            'drem_seq_d_opt': {'color': 'olive', 'linestyle': ':', 'marker': '^'},
            'drem_seq_dpp': {'color': 'teal', 'linestyle': '-', 'marker': 'D'},
            'drem_seq_cos': {'color': 'magenta', 'linestyle': '--', 'marker': 'v'},
            'adam': {'color': 'blue', 'linestyle': '-', 'marker': ''},
            'nag': {'color': 'orange', 'linestyle': '-', 'marker': ''}
        }

        for opt_name in opts_keys:
            mean_hist = np.mean(histories[opt_name], axis=0)
            # Clip NaNs and infinities to 10.0 to prevent matplotlib plotting crashes
            mean_hist_safe = np.nan_to_num(
                mean_hist, nan=10.0, posinf=10.0, neginf=10.0
            )
            markevery = max(1, len(mean_hist_safe) // 10)

            ax.plot(
                mean_hist_safe,
                label=opt_name,
                color=styles[opt_name]['color'],
                linestyle=styles[opt_name]['linestyle'],
                marker=styles[opt_name]['marker'],
                markevery=markevery,
                linewidth=1.5
            )

        ax.axhline(
            y=thr_polish,
            color='black',
            linestyle=':',
            label=f'Polish Thr (MSE < {thr_polish})'
        )

        ax.set_title(
            f"{title_prefix} | Polish={polish_optimizer} | Alpha={combo['alpha']}"
        )
        ax.set_xlabel('Epochs (0 = Init)')
        ax.set_ylabel('MSE')

        # Configure logarithmic Y-axis with human-readable decimal ticks (0.1, 0.2, 0.5, 1.0, ...)
        ax.set_yscale('log')
        ax.yaxis.set_major_locator(
            ticker.LogLocator(base=10.0, subs=(1.0, 2.0, 5.0))
        )

        def format_mse_tick(val, _):
            '''
            Formats logarithmic Y-axis tick values as standard decimal numbers.
            '''
            if val >= 10:
                return f'{val:.0f}'
            if val >= 0.1:
                return f'{val:.1f}'
            if val >= 0.01:
                return f'{val:.2f}'
            return f'{val:g}'

        ax.yaxis.set_major_formatter(ticker.FuncFormatter(format_mse_tick))
        ax.yaxis.set_minor_locator(ticker.NullLocator())

        # Set X-axis grid resolution to 1 epoch for fine-grained convergence tracking
        ax.set_xlim(0, combo['epochs'])
        ax.set_xticks(np.arange(0, combo['epochs'] + 1, 1))
        ax.tick_params(axis='x', rotation=45, labelsize=8)

        ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
        ax.grid(True, which='major', axis='x', linestyle='--', alpha=0.7)
        ax.grid(True, which='major', axis='y', linestyle='--', alpha=0.6)

        plt.tight_layout()
        params_str = '_'.join([f'{k}{v}' for k, v in combo.items()])
        fig.savefig(os.path.join(folder_name, f'exp_{i}_{params_str}.png'))
        plt.close(fig)

        # Compile summary statistics for CSV export
        res_dict = {'num': i, **combo}
        for opt in opts_keys:
            res_dict[f'mse_{opt}'] = np.mean(
                [x[f'mse_{opt}'] for x in iteration_results]
            )
            res_dict[f'mae_{opt}'] = np.mean(
                [x[f'mae_{opt}'] for x in iteration_results]
            )
            res_dict[f'time_{opt}'] = np.mean(
                [x[f'time_{opt}'] for x in iteration_results]
            )
            res_dict[f'flops_{opt}'] = flops_per_iter[opt]

            # Safely average switch epochs, avoiding empty-slice RuntimeWarnings
            if opt in switch_epochs_record:
                vals = switch_epochs_record[opt]
                if len(vals) > 0 and not np.all(np.isnan(vals)):
                    res_dict[f'switch_epoch_{opt}'] = float(np.nanmean(vals))
                else:
                    res_dict[f'switch_epoch_{opt}'] = np.nan

        return res_dict
    except Exception as e:
        print(f'Error in task {i}: {e}')
        return None

6. SUITE RUNNERS

In [13]:
def run_make_regression_suite():
    '''
    Executes the synthetic make_regression benchmark suite across multiple
    sample sizes, feature dimensions, noise levels, and step sizes.
    '''
    print('\n--- Starting Make_Regression Suite ---')
    params_grid = {
        'n_samples': [1000, 2000],
        'n_features': [5, 10, 15, 20],
        'noise': [0.1, 0.5],
        'alpha': [5e-6, 1e-5, 5e-5, 1e-4, 5e-4],
        'epochs': [30],
        'polish_optimizer': ['adam', 'nag']
    }
    folder_name = os.path.abspath('results_regression_hybrid')
    os.makedirs(folder_name, exist_ok=True)

    # Evaluate and save batch informativeness distribution on a reference synthetic dataset
    x_rep, y_rep = make_regression(
        n_samples=1000, n_features=10, noise=0.1, random_state=42
    )
    evaluate_batch_distributions(
        x_rep.astype(np.float64),
        y_rep.reshape(-1, 1).astype(np.float64),
        folder_name,
        'make_regression_N1000_D10'
    )

    keys, values = zip(*params_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    tasks = []
    for i, combo in enumerate(combinations, 1):
        x_raw, y_raw = make_regression(
            n_samples=combo['n_samples'],
            n_features=combo['n_features'],
            noise=combo['noise'],
            random_state=42
        )
        y_raw = y_raw.reshape(-1, 1).astype(np.float64)
        x_raw = x_raw.astype(np.float64)

        title = (
            f"Reg N={combo['n_samples']} "
            f"D={combo['n_features']} "
            f"Noise={combo['noise']}"
        )
        tasks.append((i, combo, folder_name, x_raw, y_raw, title))

    results = Parallel(n_jobs=8, verbose=10)(
        delayed(run_experiment)(task) for task in tasks
    )
    save_results(results, folder_name, 'results_regression_hybrid.csv')

In [14]:
def run_abalone_suite():
    '''
    Executes the benchmark suite and batch distribution analysis on the
    real-world Abalone dataset.
    '''
    print('\n--- Starting Abalone Suite ---')
    try:
        column_names = [
            'Sex', 'Length', 'Diameter', 'Height', 'Whole_weight',
            'Shucked_weight', 'Viscera_weight', 'Shell_weight', 'Rings'
        ]
        df = pd.read_csv('abalone.csv', names=column_names)
        df = df.drop(columns=['Sex'])
        y_raw = df[['Rings']].values.astype(np.float64)
        x_raw = df.drop(columns=['Rings']).astype(np.float64).values
    except Exception as e:
        print(f'Skipping Abalone: {e}')
        return

    params_grid = {
        'alpha': [5e-6, 1e-5, 5e-5, 1e-4, 5e-4],
        'epochs': [30],
        'polish_optimizer': ['adam', 'nag']
    }
    folder_name = os.path.abspath('results_abalone_hybrid')
    os.makedirs(folder_name, exist_ok=True)

    evaluate_batch_distributions(x_raw, y_raw, folder_name, 'abalone')

    keys, values = zip(*params_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    tasks = [
        (i, combo, folder_name, x_raw, y_raw, 'Abalone')
        for i, combo in enumerate(combinations, 1)
    ]
    results = Parallel(n_jobs=8, verbose=10)(
        delayed(run_experiment)(task) for task in tasks
    )
    save_results(results, folder_name, 'results_abalone_hybrid.csv')

In [15]:
def run_mpg_suite():
    '''
    Executes the benchmark suite and batch distribution analysis on the
    real-world Auto-MPG dataset.
    '''
    print('\n--- Starting Auto-MPG Suite ---')
    try:
        df = pd.read_csv('auto-mpg.csv', na_values='?')
        df = df.dropna()
        df = df.drop(columns=['car name'])
        y_raw = df[['mpg']].values.astype(np.float64)
        x_raw = df.drop(columns=['mpg']).astype(np.float64).values
    except Exception as e:
        print(f'Skipping Auto-MPG: {e}')
        return

    params_grid = {
        'alpha': [5e-6, 1e-5, 5e-5, 1e-4, 5e-4],
        'epochs': [30],
        'polish_optimizer': ['adam', 'nag']
    }
    folder_name = os.path.abspath('results_mpg_hybrid')
    os.makedirs(folder_name, exist_ok=True)

    evaluate_batch_distributions(x_raw, y_raw, folder_name, 'auto_mpg')

    keys, values = zip(*params_grid.items())
    combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

    tasks = [
        (i, combo, folder_name, x_raw, y_raw, 'Auto-MPG')
        for i, combo in enumerate(combinations, 1)
    ]
    results = Parallel(n_jobs=8, verbose=10)(
        delayed(run_experiment)(task) for task in tasks
    )
    save_results(results, folder_name, 'results_mpg_hybrid.csv')

In [16]:
def save_results(results_list, folder, filename):
    '''
    Filters out failed tasks, sorts experiment records by index, and exports
    the summary table to a CSV file.

    Args:
        results_list (list): List of result dictionaries returned by workers.
        folder (str): Target directory path.
        filename (str): Output CSV filename.
    '''
    valid = [r for r in results_list if r is not None]
    if valid:
        valid.sort(key=lambda x: x['num'])
        path = os.path.join(folder, filename)
        pd.DataFrame(valid).to_csv(path, index=False)
        print(f'Saved: {path}')


In [ ]:
run_mpg_suite()
run_abalone_suite()
run_make_regression_suite()
print('\nAll experiments completed.')